In [0]:
import time
import random
from datetime import datetime, timezone, timedelta
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

# ================= CONFIG =================
METER_COUNT = 50
TRANSFORMER_COUNT = 5
BATCH_INTERVAL_SECONDS = 30
MAX_LATENCY_SECONDS = 20
MAX_OUTAGE_BATCHES = 3
TRANSFORMER_OFFLINE_PROB = 0.03

LANDING_ZONE_METER_PATH = "/Volumes/electrical_grid/00_landing_zone/electrical_stream/meters/"
LANDING_ZONE_TRANSFORMER_PATH = "/Volumes/electrical_grid/00_landing_zone/electrical_stream/transformers/"
LANDING_ZONE_EVENTS_PATH = "/Volumes/electrical_grid/00_landing_zone/electrical_stream/grid_events/"

# ================= STATE =================
outage_tracker = {tid: 0 for tid in range(1, TRANSFORMER_COUNT + 1)}

# ================= HELPER =================
def assign_transformer(meter_id):
    return ((meter_id - 1) % TRANSFORMER_COUNT) + 1

def generate_event_id(transformer_id):
    return f"EVT_{transformer_id}_{int(time.time() * 1000)}_{random.randint(1000,9999)}"

# ================= STREAM LOOP =================
while True:

    batch_time = datetime.now(timezone.utc)

    meter_rows = []
    transformer_rows = []
    event_rows = []

    transformer_load_map = {tid: 0 for tid in range(1, TRANSFORMER_COUNT + 1)}

    # ---------- Trigger New Outages ----------
    for tid in range(1, TRANSFORMER_COUNT + 1):
        if outage_tracker[tid] == 0 and random.random() < TRANSFORMER_OFFLINE_PROB:
            outage_tracker[tid] = 1

            event_rows.append({
                "event_id": generate_event_id(tid),
                "asset_type": "transformer",
                "asset_id": tid,
                "event_type": "breaker_trip",
                "event_time": batch_time,
                "ingestion_time": batch_time
            })

    # ---------- Status Flags ----------
    transformer_status_flags = {
        tid: outage_tracker[tid] > 0 for tid in range(1, TRANSFORMER_COUNT + 1)
    }

    # ---------- Generate Meter Data ----------
    for meter_id in range(1, METER_COUNT + 1):

        transformer_id = assign_transformer(meter_id)
        is_offline = transformer_status_flags[transformer_id]

        event_time = batch_time - timedelta(seconds=random.uniform(0, MAX_LATENCY_SECONDS))

        voltage = 0.0 if is_offline else round(random.uniform(220, 240), 2)
        current = 0.0 if is_offline else round(random.uniform(2, 30), 2)

        power_kw = round((voltage * current) / 1000, 3)
        power_factor = round(random.uniform(0.92, 0.99), 3)
        energy_kwh_interval = round(power_kw * (BATCH_INTERVAL_SECONDS / 3600), 5)

        transformer_load_map[transformer_id] += power_kw

        meter_rows.append({
            "meter_id": meter_id,
            "transformer_id": transformer_id,
            "event_time": event_time,
            "voltage_v": voltage,
            "current_a": current,
            "power_kw": power_kw,
            "power_factor": power_factor,
            "energy_kwh_interval": energy_kwh_interval,
            "ingestion_time": batch_time
        })

    # ---------- Write Meter Data ----------
    spark.createDataFrame(meter_rows) \
        .write.format("delta").mode("append").save(LANDING_ZONE_METER_PATH)

    # ---------- Generate Transformer Data + Handle Recovery ----------
    for tid in range(1, TRANSFORMER_COUNT + 1):

        event_time = batch_time - timedelta(seconds=random.uniform(0, MAX_LATENCY_SECONDS))
        load_kw = round(transformer_load_map[tid], 2)

        is_offline = outage_tracker[tid] > 0

        transformer_rows.append({
            "transformer_id": tid,
            "event_time": event_time,
            "load_kw": load_kw,
            "capacity_kva": 500,
            "status": "offline" if is_offline else "online",
            "ingestion_time": batch_time
        })

        # ---------- Handle Outage Progression ----------
        if outage_tracker[tid] > 0:
            outage_tracker[tid] += 1

            # Recover after max batches
            if outage_tracker[tid] > MAX_OUTAGE_BATCHES:
                outage_tracker[tid] = 0

                event_rows.append({
                    "event_id": generate_event_id(tid),
                    "asset_type": "transformer",
                    "asset_id": tid,
                    "event_type": "breaker_reclose",
                    "event_time": batch_time,
                    "ingestion_time": batch_time
                })

    # ---------- Write Transformer Data ----------
    spark.createDataFrame(transformer_rows) \
        .write.format("delta").mode("append").save(LANDING_ZONE_TRANSFORMER_PATH)

    # ---------- Write Events ----------
    if event_rows:
        spark.createDataFrame(event_rows) \
            .write.format("delta").mode("append").save(LANDING_ZONE_EVENTS_PATH)

    print(f"Batch written at {batch_time.isoformat()}")

    time.sleep(BATCH_INTERVAL_SECONDS)

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:139)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:139)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:724)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:442)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:442)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can